## First of all we will install the required packages that will help us along the way
1. anthropic: This package will help us interact with the Anthropic API and call Claude models to generate responses.
3. PyMuPDF: This package is used for easy PDF manipulation.
4. tiktoken: This package is used to calculate the tokens in a text

### 📌 Prerequisites  

Please download and review the following documents before proceeding:  

1. **AWS1.pdf**  
   [Download Link](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing)  

2. **PROFRAC HOLDINGS, LLC Credit Agreement.pdf**  
   [Download Link](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing)  


In [ ]:
!pip install -q -U anthropic
!pip install -q -U PyMuPDF tqdm tiktoken

In [ ]:
import anthropic
import fitz
import tiktoken
import asyncio

# Make Sure to Place Your OPEN API KEY

In [ ]:
client = anthropic.Anthropic(
    api_key="Place Your OpenAi API Key Here"
)

**⚠️ Note:** **In the cell below, you need to upload a file named `AWS1.pdf`.**  
**You can download the file from the link below.**
[📥 Download AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing)/Lab-2.1(Generating-Response-without-RAG)/AWS1.pdf)


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving AWS1.pdf to AWS1.pdf


# How LLMs Behave

Imagine we are building an AI assistant that helps users understand contracts.

A user uploads a contract and asks:

> **"What is the termination notice period in this contract?"**

Before we build advanced features like RAG, agents, or tool use, let's first understand what happens when this request reaches an LLM.

We'll explore five important concepts:

1. **Tokens** — How does an LLM process text?
2. **Context Window** — How much information can the model handle at once?
3. **Sampling** — How does the model choose what to generate?
4. **Non-determinism** — Why can the same prompt produce different responses?
5. **Evaluation** — How should we test LLM responses?

Let's start by looking at how our contract is converted into **tokens**.


## Lets take a contract and try to analyse it without much instruction

First we load the PDF and extract the texts from it and generate the token count of the text

In [ ]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        pages.append({
            "page": page_number,
            "text": text
        })

    return pages

In [ ]:
short_document = extract_text("/content/AWS1.pdf")

### Combine the Contract Text

The PDF text is stored page by page in `short_document`.

Here, we combine the text from all pages into a single string so that it can be passed to Claude for analysis.

`"\n"` adds a new line between each page's text to preserve readability.

In [ ]:
contract_text = "\n".join(
    page["text"] for page in short_document
)

## Tokens

A **token** is a small piece of text that an LLM uses to process and understand language.
LLMs don't read text exactly like humans do. Before processing text, they break it into smaller pieces called tokens.


from the code below, we'll see how a real piece of our contract is converted into tokens. We will also compare the original text's number of characters with the number of tokens created by the tokenizer.

we'll use OpenAI's `cl100k_base` tokenizer to see how our contract text is converted into tokens.

#### What are we going to do?

We'll:
1. Take the text from the first page of our contract.
2. Convert the text into tokens.
3. Count the original **characters**.
4. Count the resulting **tokens**.
5. Compare both numbers.


In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")

sample = short_document[0]["text"]

# Tokenization happens here ↓
tokens = encoding.encode(sample)

print("Characters:", len(sample))
print("Tokens:", len(tokens))

Characters: 3852
Tokens: 923


**⚠️ Note:** **In the cell below, you need to upload a file named `PROFRAC HOLDINGS, LLC credit agreement.pdf`.**  
**You can download the file from the link below.**
[📥 Download PROFRAC HOLDINGS, LLC credit agreement.pdf](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing)


In [115]:
from google.colab import files
uploaded = files.upload()

Saving PROFRAC HOLDINGS, LLC credit agreement.pdf to PROFRAC HOLDINGS, LLC credit agreement (2).pdf


In [116]:
long_document = extract_text("/content/PROFRAC HOLDINGS, LLC credit agreement.pdf")

## Context Window

A **context window** is the maximum amount of tokenized information an LLM can handle in a single request.

Think of it like the model's **working memory**. Everything we send to the model needs to fit inside this space — not just the contract.

A model can only process a limited amount of information in a single request.

Let's test this with our two contracts:


In [ ]:
import base64

with open("/content/AWS1.pdf", "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

This is simply saying:
Open the PDF, read its contents, convert it into Base64, and store it in pdf_data.

```
Open the PDF → read the PDF → convert it into Base64 → store it in pdf_data.
```

Now pdf_data contains our PDF, so we send it to Claude with an instruction to analyze and summarize the contract.
We will run this request on both contracts to see what happens when the document becomes too large.

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_data
                    }
                },
                {
                    "type": "text",
                    "text": "Analyze this contract and summarize the key terms."
                }
            ]
        }
    ]
)

print(response.content[0].text)

In [ ]:
with open("/content/PROFRAC HOLDINGS, LLC credit agreement.pdf", "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

In [ ]:
try:

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_data
                        }
                    },
                    {
                        "type": "text",
                        "text": "Analyze this contract and summarize the key terms."
                    }
                ]
            }
        ]
    )

    print("✅ Contract processed successfully!")
    print(response.content[0].text)

except Exception as e:

    print("❌ Contract could not be processed")
    print()
    print("Reason: The PDF is too large for a single request.")
    print("Claude allows a maximum of 100 PDF pages for this request.")

**12-page contract**
→ Send the complete PDF to Claude
→ ✅ Successfully processed

**216-page contract**
→ Send the complete PDF to Claude
→ ❌ Request rejected because the PDF exceeds the allowed page limit

This shows an important limitation of processing large documents in a single request.

## Sampling

When an LLM generates a response, it predicts the **next token** based on the tokens that came before it.

For each next token, the model assigns different probabilities.

```text
Possible next tokens

"30"       → 60%
"thirty"   → 20%
"90"       → 10%
"one"      →  5%
"other"    →  5%
```

The model then **samples** from these possibilities.

The `temperature` parameter controls how much randomness is introduced during sampling.

```python
extra_body={"temperature": 0-1}
```

*   Low temperature → more predictable and consistent outputs
*   High temperature → more variation and less predictable outputs


High temperature → more variation and less predictable outputs

Let's ask the **same contract question 3 times** with two different temperatures and compare the results.


In [117]:
question = """
In one sentence, explain the termination notice period in the contract.
"""

### 🌡️ Temperature = 0

With a lower temperature, the model is more likely to choose the **most probable tokens**, so the responses tend to be more consistent.

In [124]:
for i in range(3):

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        system="You are a contract analysis assistant.",
        # 🌡️ Temperature controls response variation
        extra_body={"temperature": 0},
        messages=[
            {
                "role": "user",
                "content": question + "\n\n" + small_text
            }
        ]
    )

    print("----- Response", i + 1, "-----")
    print(response.content[0].text)

----- Response 1 -----
# Termination Notice Period

Either party may terminate this Agreement for convenience with at least 30 days' advance notice, or for cause if the other party materially breaches the Agreement and fails to cure within 30 days of receiving notice, though AWS may terminate immediately for certain conditions including security risks, payment breaches, or legal compliance requirements.
----- Response 2 -----
# Termination Notice Period

Either party may terminate this Agreement for convenience with at least 30 days' advance notice, or for cause if the other party materially breaches the Agreement and fails to cure within 30 days of receiving notice, though AWS may terminate immediately for certain conditions including security risks, payment breaches, or legal compliance requirements.
----- Response 3 -----
# Termination Notice Period

Either party may terminate this Agreement for convenience with at least 30 days' advance notice, or for cause if the other party mater

### 🔥 Temperature = 1

With a higher temperature, the model allows **more variation** when choosing tokens, so the same question can produce different responses.

In [125]:
for i in range(3):

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        system="You are a contract analysis assistant.",
        # 🌡️ Temperature controls response variation
        extra_body={"temperature": 1},
        messages=[
            {
                "role": "user",
                "content": question + "\n\n" + small_text
            }
        ]
    )

    print("----- Response", i + 1, "-----")
    print(response.content[0].text)

----- Response 1 -----
# Termination Notice Period

Either party may terminate this Agreement for convenience by providing at least 30 days' advance notice, while termination for cause requires 30 days to cure a material breach after receiving notice, though AWS may terminate immediately for security violations or legal compliance reasons.
----- Response 2 -----
# Termination Notice Period

Either party may terminate this Agreement for convenience with at least 30 days' advance notice, or for cause if the other party materially breaches the Agreement and fails to cure within 30 days of receiving notice.
----- Response 3 -----
# Termination Notice Period Summary

Either party may terminate the agreement for convenience with at least 30 days' advance notice, or for cause if the other party materially breaches the agreement and fails to cure within 30 days of notice, though AWS may terminate immediately for certain violations including security risks, payment defaults, or legal compliance

### 🔍 What did we observe?

* **Higher temperature** → more randomness and potentially more varied responses
* **Lower temperature** → more predictable and consistent responses
* The underlying model is still performing the same task; the difference is in how tokens are selected during generation.

💡 **Key takeaway:** LLM outputs are probabilistic. Therefore, the same prompt does not always guarantee the exact same wording.


### 🔄 From Sampling to Non-Determinism

Look at the three responses we generated using the **same question and the same contract**.

At a higher temperature, the model may make different token choices, which can lead to different responses.

This is called **non-determinism**:

> **The same input does not necessarily produce the exact same output every time.**

For example:

```text
Run 1 → "The termination notice period is 30 days."
Run 2 → "The contract requires 30 days' notice for termination."
Run 3 → "Either party must provide 30 days notice before termination."
```

The wording is different, but the **meaning is the same**.

💡 This creates an important challenge when testing LLM applications:
**We shouldn't always test whether the output is exactly the same — we should test whether the output is correct.**


### 🧪 How Should We Test LLM Applications?

Traditional software often uses **exact-match testing**:

```text
Expected output == Actual output
```

For LLM applications, this can be too strict because multiple responses can be correct.

Instead, we can evaluate:

* **Correctness** — Did the model identify the right term?
* **Relevance** — Did it answer the question?
* **Completeness** — Did it include the important information?
* **Grounding** — Is the answer supported by the contract?

So instead of asking:

> "Did the model produce exactly this sentence?"

we ask:

> **"Did the model produce a correct answer?"**

This is one of the key differences between testing traditional software and testing LLM applications.


# Model & Reasoning

So far, we've seen how an LLM processes tokens, works within a context window, and generates probabilistic responses.

Now let's ask two important questions:

1. **Which model should we use?**
2. **When does a task require deeper reasoning?**

not every task needs the same level of intelligence.

For example:

* Extracting a contract date → relatively simple
* Summarizing a clause → moderate
* Determining whether a termination clause creates a business risk → more complex

Let's see how **model selection** and **reasoning effort** affect our application.


## 🎯 Choosing the Right Model

Different models offer different trade-offs between:

* **Capability** — How well the model handles complex tasks
* **Latency** — How quickly it responds
* **Cost** — How expensive each request is

There is no single "best" model for every task.

Instead, we choose a model based on what our application actually needs.


### ⚖️ Model Trade-offs

| Model      | Capability | Latency   | Cost      |
| ---------- | ---------- | --------- | --------- |
| **Haiku**  | Medium     | 🟢 Low    | 🟢 Low    |
| **Sonnet** | High       | 🟡 Medium | 🟡 Medium |
| **Opus**   | Very High  | 🔴 Higher | 🔴 Higher |

💡 **Key idea:** Choose the model based on the task — not every task needs the most capable model.


## Reasoning

**Reasoning = giving the model more time/tokens to think through complex tasks.**

* **Low effort** → faster + cheaper
* **High effort** → deeper reasoning + potentially higher cost/latency
* **✨ Adaptive thinking** → the model decides when more reasoning is useful

### ✨ Adaptive Thinking

With **adaptive thinking**, we don't manually decide how much reasoning the model should use.

Instead, Claude dynamically decides **when additional reasoning is needed** based on the complexity of the task.

In our example, adaptive thinking is enabled here:

```python
thinking={"type": "adaptive"}
```

We also set:

```python
output_config={"effort": "high"}
```

This tells Claude to use a **high reasoning effort**, while adaptive thinking allows the model to determine how much reasoning is actually useful.

👀 **Now let's see this in action.**

Run the prompt below and inspect the response.

You will see that Claude can return separate **thinking** and **text** blocks:

* 🧠 **Thinking block** → reasoning content returned by the API
* 💬 **Text block** → Claude's final answer

This lets us observe how **adaptive thinking** is being used for a complex contract-analysis task.


In [126]:
from IPython.display import display, Markdown

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    # ✨ Adaptive Thinking: Claude decides when reasoning is useful
    thinking={"type": "adaptive"},
    # Ask Claude to use high reasoning effort
    output_config={"effort": "high"},
    messages=[
        {
            "role": "user",
            "content": f"""
You are a contract risk analyst.

Analyze the following AWS Customer Agreement and determine:

1. What termination rights does each party have?
2. What notice periods apply?
3. What happens to the customer's access and content after termination?
4. Does this create any significant risk for the customer?
5. Give a concise recommendation to the contract reviewer.

Support your conclusion with the relevant section numbers.

Contract:

{contract_text}
"""
        }
    ]
)

# Display Claude's reasoning and final answer separately
for block in response.content:

    if block.type == "thinking":
        display(Markdown(
            f"""
### 🧠 Claude's Reasoning

{block.thinking}
"""
        ))

    elif block.type == "text":
        display(Markdown(
            f"""
### 💬 Final Answer

{block.text}
"""
        ))


### 🧠 Claude's Reasoning

Let me analyze this AWS Customer Agreement systematically, focusing on the termination-related provisions and associated risks for the customer (XYZ Software Solutions).

Let me identify the key sections:

1. **Termination Rights** - Section 5
2. **Notice Periods** - Various sections
3. **Post-termination access and content** - Section 5.3
4. **Customer risks** - Multiple sections
5. **Recommendation**

Let me go through each question:

### 1. Termination Rights

**Customer's Rights:**
Under Section 5.2(a) the customer can terminate for convenience by giving notice and closing their account, or terminate for cause under 5.2(b)(i) if AWS commits an uncured material breach after 30 days' notice. AWS has mirroring rights, but also holds a broader right under 5.2(a) to terminate for any reason with 30 days' notice, plus an immediate termination right under 5.2(b)(ii) tied to suspension triggers like security risks or breaches under Section 4 -- this asymmetry is worth flagging as a risk area.

I'm also noting other notice requirements: 12 months for service discontinuation, 90 days for adverse SLA changes, and 30 days for fee increases, while agreement modifications take effect upon posting. For post-termination effects, all customer rights end immediately, fees remain owed through the termination date, AWS content must be returned or destroyed, and certain sections survive termination.

Within the standard 30-day window after termination, AWS won't remove customer content and will allow retrieval if all fees are paid—but critically, this protection doesn't apply if AWS terminated for cause under Section 5.2(b), meaning customers could lose access to their content immediately in that scenario. This is a major risk: AWS's broad right to terminate immediately without meaningful notice, including for reasons entirely outside the customer's control like third-party partner changes or legal compliance.

I'm also noting the asymmetry in termination rights—customers must close their account when terminating while AWS only gives 30 days' notice—and the conditional nature of content retrieval, where any payment dispute could block a customer from getting their data back. Additionally, Section 4's suspension rights are extremely broad, letting AWS suspend service immediately upon notice while the customer still owes fees and receives no SLA credits during that suspension.

Beyond that, I'm flagging AWS's unilateral right to modify the agreement anytime by posting changes online, its exclusion of liability for damages from termination or suspension, the fixed 12-month term without auto-renewal, AWS's ability to assign the agreement without consent, and the risky provision letting AWS terminate immediately if a third-party partner relationship changes—something entirely outside the customer's control.

Additional concerns include mandatory binding arbitration with waived jury trial and class action rights, AWS's ability to raise fees with just 30 days' notice, customer's sole responsibility for backups under Section 2.3 combined with the danger of losing data access upon immediate for-cause termination, and the need to plan around the March 31, 2024 expiration since the contract won't renew automatically.

For recommendations, I want to suggest negotiating a minimum notice period for convenience-based termination, securing a guaranteed content retrieval window even in for-cause scenarios, ensuring retrieval isn't blocked by payment disputes, planning ahead for the fixed-term expiration, and building a strong backup strategy given the customer's exposure under Section 2.3.

I'm also noticing a date inconsistency - the header lists "Last Updated: April 20, 2023" while the effective date is stated as April 1, 2023, with the term running twelve months to March 31, 2024, and Section 5.2(b)(ii) explicitly confirming no automatic renewal. Beyond that, Section 10 gives AWS sweeping unilateral rights to modify the agreement just by posting changes, and Section 11.4's choice of Washington State law could be inconvenient depending on where XYZ Software Solutions is based.

Now I'm ready to pull this all together into a clear, organized analysis.





### 💬 Final Answer

# Contract Risk Analysis: AWS Customer Agreement
**Customer:** XYZ Software Solutions | **Contract Period:** April 1, 2023 – March 31, 2024 | **Contract Value:** USD 35,000

---

## 1. Termination Rights of Each Party

### Customer's Rights
| Right | Basis | Conditions |
|---|---|---|
| Terminate for Convenience | §5.2(a)


# Prompting: Zero-Shot, One-Shot & Few-Shot

**Prompting is the process of giving an LLM instructions, context, and examples to guide it toward the desired output.**

In this, we will use a **contract PDF** and ask Claude to extract important contract terms.

We will progressively add examples to the prompt and observe how the output changes.

### Zero-Shot
No examples are provided.

**Task + Context**

Use when the task is simple and the model can infer the expected behavior.

### One-Shot
One example is provided.

**Task + Example + Context**

Use when we want to show the model the expected output format or style.

### Few-Shot
Multiple examples are provided.

**Task + Examples + Context**

Use when the task has different cases or edge cases that are difficult to describe using instructions alone.

> More examples do not always mean better results.
> The goal is to provide **just enough guidance** while keeping the prompt small.

We will compare the outputs and see whether additional examples improve the result.

#Zero-Shot Prompt

In [ ]:
prompt = f"""
Extract the key terms from this contract.

Return:
- Parties
- Effective Date
- Expiration Date
- Contract Value
- Payment Terms
- Termination Terms
- Governing Law

Contract:
{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)

# AWS Customer Agreement - Key Terms Extract

## Parties
- **Provider:** Amazon Web Services (AWS) and its applicable regional contracting party (based on Account Country)
- **Customer:** XYZ Software Solutions

## Effective Date
- **April 1, 2023** (or upon first use of Services, whichever is earlier)

## Expiration Date
- **March 31, 2024** (12-month contract duration)
- **Non-renewable** - Contract does not automatically renew after end date

## Contract Value
- **USD 35,000** paid annually before the start of work
- *Note: For India customers, fees displayed in USD but invoiced in INR (Indian Rupees) at conversion rate determined on invoice date*

## Payment Terms
- **Billing Frequency:** Monthly calculation and billing
- **Payment Method:** One of AWS's supported payment methods required
- **Payment Requirements:** 
  - Without setoff or counterclaim
  - Without deduction or withholding
  - Gross (before withholding taxes)
- **Price Increases:** 30 days' advance notice required fo

#One-Shot Prompt

In [ ]:
prompt = f"""
Extract the key terms from the contract.

Follow the output format shown in this example.

Example:

Contract:
"ABC Corp and XYZ Ltd entered into an agreement
effective January 1, 2025 for $50,000."

Output:
{{
    "parties": ["ABC Corp", "XYZ Ltd"],
    "effective_date": "2025-01-01",
    "contract_value": 50000
}}

Now extract the key terms from this contract:

{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)

```json
{
    "parties": ["Amazon Web Services (AWS)", "XYZ Software solutions"],
    "effective_date": "2023-04-01",
    "contract_value": 35000,
    "contract_value_currency": "USD",
    "payment_terms": "annually",
    "contract_duration_months": 12,
    "termination_date": "2024-03-31",
    "auto_renewal": false,
    "billing_frequency": "monthly",
    "late_payment_interest_rate": "1.5% per month",
    "service_discontinuation_notice_period_days": 365,
    "service_level_agreement_change_notice_days": 90,
    "termination_for_convenience_notice_days": 30,
    "termination_for_cause_cure_period_days": 30,
    "post_termination_data_retention_days": 30,
    "confidentiality_period_years": 5,
    "data_breach_notification_hours": 48,
    "governing_law": "State of Washington",
    "dispute_resolution": "Binding Arbitration",
    "arbitration_provider": "American Arbitration Association (AAA)",
    "arbitration_fee_reimbursement_threshold": 10000,
    "liability_cap": "Amount paid for

#Few-Shot Prompt

In [ ]:
prompt = f"""
Extract key terms from the contract.

Use these examples to understand how different
types of information should be represented.

Example 1 — Text:

Contract:
"The agreement is between ABC Corp and XYZ Ltd."

Output:
{{
    "parties": ["ABC Corp", "XYZ Ltd"]
}}


Example 2 — Number:

Contract:
"The total contract value is USD 50,000."

Output:
{{
    "contract_value": 50000
}}


Example 3 — Date:

Contract:
"The agreement becomes effective on January 1, 2025."

Output:
{{
    "effective_date": "2025-01-01"
}}


Example 4 — Missing value:

Contract:
"The agreement becomes effective on January 1, 2025."

Output:
{{
    "effective_date": "2025-01-01",
    "expiration_date": null
}}


Now apply these examples to the following contract:

{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.content[0].text)

```json
{
  "contract_type": "AWS Customer Agreement",
  "parties": [
    "Amazon Web Services",
    "XYZ Software solutions"
  ],
  "effective_date": "2023-04-01",
  "expiration_date": "2024-03-31",
  "contract_duration_months": 12,
  "contract_value": 35000,
  "contract_value_currency": "USD",
  "contract_value_payment_schedule": "annually",
  "last_updated": "2023-04-20",
  "auto_renewal": false,
  "key_terms": {
    "service_fees": "Monthly billing for Services",
    "payment_methods": "Supported payment methods on AWS Site",
    "late_payment_interest": "1.5% per month",
    "fee_increase_notice": "30 days minimum notice",
    "termination_for_convenience_notice": "30 days minimum notice",
    "termination_for_cause_cure_period": "30 days",
    "post_termination_content_retrieval": "30 days",
    "confidentiality_period": "5 years post-termination"
  },
  "governing_law": "Laws of the State of Washington",
  "dispute_resolution": "Binding arbitration",
  "arbitration_administrator

## What did we observe?

| Prompt | Examples | Purpose |
|---|---:|---|
| Zero-shot | 0 | Let the model infer the task |
| One-shot | 1 | Demonstrate the expected format |
| Few-shot | 2–4 | Demonstrate different cases and edge cases |

The important idea is **not**:

> "Add more examples whenever the output is bad."

Instead, ask:

> "What is the smallest amount of guidance that gives us the quality we need?"

If adding more examples makes the prompt huge, we should also consider changing the **model, context, reasoning effort, or output constraints** rather than creating a 150-page prompt.

# Claude API: Sync, Streaming, Async & Batching

The Claude API can be used in different ways depending on how we want our application to handle responses.

- **Synchronous:** Send a request and wait until the complete response is returned.
- **Streaming:** Receive the response progressively as Claude generates it.
- **Async:** Start requests without blocking the application while waiting for results.
- **Batching:** Submit many requests together, which is useful for evaluations and large-scale processing.

In this lab, we will use a contract-analysis task and see how each approach works.

Normal:

```
Request A
   ↓
wait
   ↓
Response A

Request B
   ↓
wait
   ↓
Response B
```



v/s

```

Async:

Request A ────────────────┐
Request B ──────────┐     │
Request C ───────┐  │     │
                 ↓  ↓     ↓
              Claude API
                 ↓
             Responses
```

#Synchronous request

In [ ]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    # Send the contract and instructions to Claude
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this contract and identify:

1. Contract parties
2. Effective date
3. Expiration date
4. Contract value
5. Termination terms

Contract:
{contract_text}
"""
        }
    ]
)

# This line runs only after Claude has returned the response
print(response.content[0].text)

# AWS Customer Agreement Analysis

## 1. Contract Parties

| Party | Details |
|-------|---------|
| **Service Provider** | Amazon Web Services (AWS) / applicable AWS Contracting Party per Section 12 |
| **Customer** | XYZ Software Solutions |

> **Note:** The specific AWS Contracting Party depends on the customer's Account Country. For India-based customers, the contracting party would be **Amazon Web Services India Private Limited**.

---

## 2. Effective Date

**April 1, 2023**

> Per the agreement: *"This Agreement takes effect on 1st April 2023 or, if earlier, when you use any of the Services."*

---

## 3. Expiration Date

**March 31, 2024**

> Per the agreement: *"Contract is valid for a duration of 12 months and will end on 31st March 2024."*

⚠️ **Important:** The contract **does not auto-renew**, as explicitly stated in Section 5.2(b)(ii): *"This contract does not renew automatically after it reaches the end date of contract."*

---

## 4. Contract Value

**USD $35,000 annual

### What happened?

The request blocks until Claude finishes generating the complete response.

**Request → Wait → Complete Response**

#Streaming response

In the previous example, we made a synchronous API call and waited for Claude to return the complete response.

With streaming, Claude sends the response piece by piece as it is generated.

Instead of waiting for the entire answer, we can display each piece immediately.

###🔄 How Streaming Works

When we use:
```
client.messages.stream(...)

Claude starts generating the response and sends the generated text in small chunks.

Our code then reads those chunks using:

for text in stream.text_stream
```

In [ ]:
with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this contract and identify the key termination
rights and notice periods.

Contract:
{contract_text}
"""
        }
    ]
) as stream:

    for text in stream.text_stream:
        print(text, end="", flush=True)

# Contract Termination Rights & Notice Periods Analysis

## AWS Customer Agreement — XYZ Software Solutions
**Contract Period:** April 1, 2023 – March 31, 2024 | **Value:** USD 35,000

---

## 1. TERMINATION RIGHTS SUMMARY

### 1.1 Termination for Convenience (Section 5.2a)

| Party | Right | Notice Required |
|-------|-------|-----------------|
| **Customer (XYZ)** | May terminate for **any reason** | Must provide notice + close all service accounts |
| **AWS** | May terminate for **any reason** | Minimum **30 days' advance notice** |

> ⚠️ **Key Asymmetry:** XYZ Software Solutions can terminate without a defined notice period beyond closing accounts, while AWS must provide 30 days' notice.

---

### 1.2 Termination for Cause (Section 5.2b)

#### By Either Party
- **Trigger:** Material breach of Agreement
- **Cure Period:** **30 days** from receipt of written notice
- **Condition:** Breach must remain uncured after the 30-day cure period
- **Customer Obligation:** Must close account n

### What changed?

With streaming, we don't wait for the entire response.

Claude sends generated text incrementally:

**Request → Token/Chunk → Token/Chunk → Token/Chunk → ... → Complete**

This is useful for chat interfaces where we want the user to see the response immediately.

#Async / non-blocking request

## Asynchronous API Call

We can also call Claude **asynchronously** using `AsyncAnthropic`.

Unlike a synchronous call, an asynchronous call allows Python to work with other tasks while waiting for Claude's response.

### 🔑 Authentication

Because we are using Anthropic's API, we need an **Anthropic API key**.

Make sure you replace the placeholder below with your Anthropic API key.

> ⚠️ Do not use an OpenAI API key here. `AsyncAnthropic` requires an Anthropic API key.

### 🔄 Execution Flow

```text
Start async function
       ↓
Send request to Claude
       ↓
      await ⏳
       ↓
Receive response
       ↓
Return result
```

The important part is:

```python
response = await async_client.messages.create(...)
```

The `await` tells Python to **wait for this asynchronous operation to complete** before using the response.

We use `async def` to define the asynchronous function and `await` when making the API call.


In [ ]:
import asyncio
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic(
    api_key="Place Your OpenAi API Key Here"
)

async def analyze_contract():
    response = await async_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""
Analyze this contract and summarize the termination
rights and associated risks.

Contract:
{contract_text}
"""
            }
        ]
    )

    return response.content[0].text


result = await analyze_contract()

print(result)

# AWS Customer Agreement: Termination Rights & Associated Risks Analysis

**Contract Parties:** Amazon Web Services (AWS) & XYZ Software Solutions
**Contract Period:** April 1, 2023 – March 31, 2024
**Contract Value:** USD 35,000 (paid annually)

---

## 1. TERMINATION RIGHTS SUMMARY

### 1.1 Termination for Convenience (Section 5.2a)

| Party | Rights | Notice Required |
|-------|--------|-----------------|
| **XYZ Software Solutions** | May terminate for **any reason** | Must provide notice AND close all service accounts |
| **AWS** | May terminate for **any reason** | Minimum **30 days** advance notice required |

> **Key Imbalance:** XYZ must close all accounts to complete termination, while AWS only needs to provide 30 days notice — creating an **asymmetric burden** on XYZ.

---

### 1.2 Termination for Cause (Section 5.2b)

#### Either Party May Terminate When:
- The other party commits a **material breach**
- The breach remains **uncured for 30 days** after written notice
- XYZ 

### Why async?

Async allows our application to start an API call without blocking other work.

This becomes especially useful when we have multiple independent tasks that can run at the same time.

#Multiple async calls

In [ ]:
async def ask_claude(prompt):
    response = await async_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text


prompts = [
    "Identify the contract parties.",
    "Identify the contract value.",
    "Identify the effective date.",
    "Identify the termination terms.",
    "Identify the governing law."
]

results = await asyncio.gather(
    *(ask_claude(prompt + f"\n\nContract:\n{contract_text}")
      for prompt in prompts)
)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(result)


--- Result 1 ---
# Contract Parties

Based on the AWS Customer Agreement, the two contracting parties are:

---

## Party 1: AWS (Service Provider)
- **Legal Name:** Amazon Web Services (AWS), referred to as "we," "us," or "our"
- **Applicable Contracting Entity:** The specific AWS Contracting Party is determined based on the customer's Account Country (as detailed in Section 12)
- **Role:** Provider of cloud services and infrastructure

---

## Party 2: Customer (Service Recipient)
- **Legal Name:** **XYZ Software Solutions**
- **Referred to as:** "you" or "your"
- **Role:** Purchaser/user of AWS cloud services

---

## Key Agreement Details

| Detail | Information |
|--------|-------------|
| **Effective Date** | April 1, 2023 |
| **End Date** | March 31, 2024 |
| **Contract Duration** | 12 months |
| **Contract Value** | USD 35,000 (paid annually) |
| **Last Updated** | April 20, 2023 |

---

> **Note:** The specific AWS legal entity that serves as the contracting party may vary de

## Batching Contract Evals

When evaluating a model across multiple contracts, we may want to ask the **same set of questions** for every document.

Here, we have two contracts:

* 📄 **Short contract** — 12 pages
* 📚 **Long contract** — 216 pages

We will ask the same **3 questions** about both contracts.

This creates:

**2 contracts × 3 questions = 6 independent requests**

Instead of sending each request separately, we can submit all 6 requests together using an **Anthropic Message Batch**.

### Workflow

```text
12-page contract  ──┐
                    ├── 3 questions each ──→ 6 requests
216-page contract ──┘                           │
                                               ↓
                                          Message Batch
                                               ↓
                                         Check Status
                                               ↓
                                        Retrieve Results
```

Batching is useful for **evals**, where we often need to run the same test cases across many documents or prompts.

In this example, we can also compare how the model answers the same questions when working with a **short vs. long contract**.


### 1. Prepare the two contracts + 3 common questions

In [140]:
short_contract = "\n".join(page["text"] for page in short_document)
long_contract = "\n".join(page["text"] for page in long_document)

questions = [
    "What is the total contract value?",
    "What are the payment terms?",
    "What are the termination conditions?"
]

### 2. Create 6 requests and submit them as one batch

In [141]:
requests = []

for name, contract in [
    ("12_page_contract", short_contract),
    ("216_page_contract", long_contract)
]:
    for i, question in enumerate(questions, 1):
        requests.append({
            "custom_id": f"{name}_q{i}",
            "params": {
                "model": "claude-sonnet-4-6",
                "max_tokens": 500,
                "messages": [{
                    "role": "user",
                    "content": f"""
Analyze this contract and answer the question.

Contract:
{contract}

Question: {question}

Answer only using information from the contract.
"""
                }]
            }
        })

# 🚀 SEND ALL 6 REQUESTS TO ANTHROPIC AS ONE BATCH
batch = client.messages.batches.create(requests=requests)

print(f"Submitted {len(requests)} requests")
print(f"Batch ID: {batch.id}")

Submitted 6 requests
Batch ID: msgbatch_01T8cADShtxMAptSswW5fDs6


### 3. Wait for completion + retrieve all answers

⚠️ Note: Batch processing is asynchronous and may take some time to complete.

In [142]:
import time

while True:
    status = client.messages.batches.retrieve(batch.id)

    print(f"Batch status: {status.processing_status}")

    if status.processing_status == "ended":
        break

    time.sleep(2)

print("\n✅ Batch processing complete!")

for result in client.messages.batches.results(batch.id):
    if result.result.type == "succeeded":
        print(f"\n{result.custom_id}")
        print(result.result.message.content[0].text)

Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status

### Why Use Message Batches?

You might be wondering:

> **"Couldn't we just use asynchronous calls to send all 6 requests at once?"**

Yes! We could.

The difference is **what we're optimizing for**.

With asynchronous calls, our application starts multiple requests concurrently and waits for their responses. This is useful when we need the results relatively quickly.

With a **Message Batch**, we submit a collection of independent requests as a background job. We don't need an immediate response. We can check the batch status later and retrieve the results when processing is complete.

For our contract example:

**2 contracts × 3 questions = 6 independent requests**

Instead of managing those six requests individually, we submit them as **one batch**.

### When should you use each?

* **Synchronous call** → "I need this answer now."
* **Async calls** → "I need several answers concurrently."
* **Message Batch** → "I have many independent jobs and can process them in the background."